In [ ]:
#pip install langchain langchain-core langchain-groq groq python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [ ]:
# Load environment variables
load_dotenv()

# Initialize Groq chat
groq_chat = ChatGroq(
    temperature=0.7,
    model_name="llama-3.1-8b-instant",  # Groq's model
    api_key=os.getenv("GROQ_API_KEY")
)

# Simple query
response = groq_chat.invoke("Explain quantum computing in simple terms")
print(response.content)

# Building a Simple Chat Application

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor. Respond to student questions clearly and concisely."),
    ("human", "{input}")
])

In [ ]:
# Set up the chain
model = ChatGroq(temperature=0.5, model_name="llama-3.1-8b-instant")
chain = prompt | model

# Interactive chat
print("AI Tutor: Hello! Ask me anything. Type 'quit' to exit.")
while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        break
    response = chain.invoke({"input": user_input})
    print(f"AI Tutor: {response.content}")

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [ ]:
# Sample documents (in a real project, load from files)
documents = [
    "The Apollo 11 mission landed humans on the Moon in 1969. Neil Armstrong and Buzz Aldrin were the first to walk on the lunar surface.",
    "SpaceX, founded by Elon Musk, launched the Falcon Heavy rocket in 2018, one of the most powerful rockets in history.",
    "The International Space Station (ISS) is a collaborative project between NASA, ESA, JAXA, CSA, and Roscosmos, orbiting Earth since 1998."
]

In [ ]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.create_documents(documents)

In [ ]:
chunks

In [ ]:
# Create embeddings and store in ChromaDB
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embedding_function)

In [ ]:
# Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [ ]:
relevant_chunks = retriever.get_relevant_documents("What is the capital of France?")
relevant_chunks

In [ ]:
context = " ".join([chunk.page_content for chunk in relevant_chunks])
context

In [ ]:
# Function to answer questions
def answer_question(question):
    # Retrieve relevant chunks
    relevant_chunks = retriever.get_relevant_documents(question)
    context = " ".join([chunk.page_content for chunk in relevant_chunks])
    
    # Create prompt
    prompt = f"""Context: {context}
Question: {question}
Answer based on the context provided. If the context doesn't contain the answer, say so."""
    
    # Generate response using Groq
    response = model.invoke(prompt)
    return response.content


In [ ]:
questions = [
        "Who were the first humans to walk on the Moon?",
        "What is the International Space Station?",
        "What is the capital of France?"
    ]

In [ ]:
for q in questions:
    print(f"Q: {q}")
    print(f"A: {answer_question(q)}\n")